# Analysis Experiments

Use this notebook to test different prompts/models for generating call analysis.

## Quick Notes
- **Topics & keywords are auto-created** if they don't exist in the database
- **Everything is normalized to lowercase** ("Billing" → "billing")
- **Tweak the prompt** in section 2 to improve results
- **Compare models** by uncommenting section 4

## Workflow
1. Add your OpenAI API key below - you can experiment with open source models also, this was just easier
2. Load a transcript (from sample file or database)
3. Tweak the prompt and run analysis
4. Compare results
5. Save to database when happy

In [ ]:
import json
import os
import sys
from pathlib import Path

# Setup paths
backend = Path.cwd().parent.parent
sys.path.insert(0, str(backend))

from openai import OpenAI

# ========== CONFIG ==========
OPENAI_API_KEY = ""  # Paste your key here, or leave empty to use .env
# ============================

if not OPENAI_API_KEY:
    from dotenv import load_dotenv
    load_dotenv(backend.parent / ".env")

openai_client = OpenAI(api_key=OPENAI_API_KEY or None)

## 1. Load a transcript

In [ ]:
# Option A: Load from sample file (has nested "turns" structure)
with open(backend / "transcripts" / "positive.json") as f:
    samples = json.load(f)

sample = samples[0]
transcript = sample["transcript"]["turns"]  # Sample files use {"transcript": {"turns": [...]}}

print(f"Loaded transcript with {len(transcript)} turns")
print(f"First turn: {transcript[0]}")

In [ ]:
# Option B: Load from database (flat array format)
# from supabase import create_client
# from database.constants import Tables
#
# supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])
# result = supabase.table(Tables.CALLS).select("id, transcript").limit(1).execute()
# call = result.data[0]
# call_id = call["id"]
# transcript = call["transcript"]  # Database stores as [{"speaker": "...", "text": "..."}, ...]

## 2. Define the analysis prompt

Tweak this to experiment with different prompts.

**Required output fields:** `summary`, `sentiment_score`, `sentiment_label`, `key_moves`, `is_resolved`, `topics`, `keywords`

In [ ]:
SYSTEM_PROMPT = """
You are a call center analyst. Analyze the following call transcript and return a JSON object with:

- summary: Brief summary of the call (2-3 sentences)
- sentiment_score: Float from -1.0 (very negative) to 1.0 (very positive)
- sentiment_label: One of "positive", "neutral", "negative"
- key_moves: List of effective techniques the agent used (e.g., "Acknowledged frustration", "Offered solution")
- is_resolved: Boolean - was the customer's issue resolved?
- topics: List of topics discussed (e.g., "billing", "refund", "technical support")
- keywords: List of important keywords from the conversation

Return ONLY valid JSON, no other text.
""".strip()

MODEL = "gpt-4o-mini"  # Change to test different models

In [ ]:
def format_transcript(turns: list[dict]) -> str:
    """Format transcript turns for the prompt."""
    return "\n".join(f"{t['speaker']}: {t['text']}" for t in turns)

def analyze_transcript(turns: list[dict], model: str = MODEL) -> dict:
    """Run analysis on a transcript. Returns parsed JSON result."""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_transcript(turns)}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

## 3. Run analysis

In [ ]:
result = analyze_transcript(transcript)
print(json.dumps(result, indent=2))

## 4. Compare models (optional)

Run the same transcript through different models to compare.

In [ ]:
# models_to_test = ["gpt-4o-mini", "gpt-4o"]
# 
# results = {}
# for model in models_to_test:
#     print(f"Running {model}...")
#     results[model] = analyze_transcript(transcript, model=model)
# 
# # Compare
# for model, res in results.items():
#     print(f"\n=== {model} ===")
#     print(f"Sentiment: {res['sentiment_score']} ({res['sentiment_label']})")
#     print(f"Topics: {res['topics']}")
#     print(f"Key moves: {res['key_moves']}")

## 5. Save to database

When you're happy with the results, uncomment and run this cell.

**What happens:**
- Analysis is saved to `call_analyses` table
- Topics/keywords are created if they don't exist, then linked to the analysis

In [ ]:
# from supabase import create_client
# from database import analysis
# 
# supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])
# 
# # Save analysis (use call_id from step 1 if loaded from database)
# call_id = "your-call-id-here"
# 
# saved = analysis.create_analysis(
#     supabase,
#     call_id=call_id,
#     summary=result["summary"],
#     sentiment_score=result["sentiment_score"],
#     sentiment_label=result["sentiment_label"],
#     key_moves=result["key_moves"],
#     is_resolved=result["is_resolved"],
# )
# 
# # Add topics and keywords
# analysis.add_topics_to_analysis(supabase, saved["id"], result["topics"])
# analysis.add_keywords_to_analysis(supabase, saved["id"], result["keywords"])
# 
# print(f"Saved analysis: {saved['id']}")